# MorphoNup — modality analysis

Which inputs matter? Input set: sequence only (no shape anywhere), shape only (no sequence routes), shuffled shape (MorphoNup unchanged, but every sequence gets another sequence's shape tracks). Shape preprocessing: without the PLS reduction (same 10 tracks), all 14 tracks unreduced (no preprocessing at all), and without EP. All arms are retrained with the same splits, seeds and recipe and compared with MorphoNup on the same outer test rows.

Key contrasts: MorphoNup − sequence only = value of shape; − shape only = value of sequence; − shuffled shape = value of the shape information beyond its extra parameters (shape is computed from the sequence); − no PLS = value of the reduction; − all 14 tracks = value of the whole shape preprocessing; − no EP = value of EP.

Statistics, figures and tables as in the architecture analysis (forest plot `deltas`, box plot `boxplot_accuracy`, heatmap, metric panel, discordance, ROC/PR, training curves; `manuscript_table`, `metrics_table`), plus `roc_pr_arms` (ROC and precision–recall curves of MorphoNup and the three input-set variants in one panel).

`RUN_MODE = "background"` trains the missing arms one after another in a single detached job (MorphoNup itself must already be trained for this dataset and k, with the same seeds). Figures and tables are refreshed automatically whenever an arm finishes a seed; the Results cells display them.

In [ ]:
DATASET = "DatasetNup_1/Homo_sapiens"
K_FOLDS = 5
SEEDS = [151000, 152000, 153000]          # the reported seeds (config.SEEDS)
GROUP = "modality"
ARMS = None                               # None = every arm of the group; or a list such as ["no_kmer"]
RUN_MODE = "results_only"                # "background" | "inline" | "results_only"
CHAINS = 2                                # parallel jobs; each trains its share of the arms one after another
ALLOW_BUSY_GPU = False

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
_null, _err = os.open(os.devnull, os.O_WRONLY), os.dup(2)
os.dup2(_null, 2)                                    # TensorFlow's CUDA start-up lines go to C-level stderr
try:
    import tensorflow  # noqa: F401
finally:
    os.dup2(_err, 2)
    os.close(_null)
    os.close(_err)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "morphonup" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display

from morphonup import config, figures, jobs
from morphonup.cv import run_dir

pd.set_option("display.width", 200, "display.max_columns", 40)


def show(path, width=900):
    """Display a saved figure with its caption (from the folder's figure_catalog.csv)."""
    path = Path(path)
    if not path.exists():
        print("missing:", path.relative_to(ROOT))
        return
    cat = path.parent / "figure_catalog.csv"
    caption = pd.read_csv(cat).set_index("png").suggested_caption.get(path.name, "") if cat.exists() else ""
    display(Markdown(f"**{path.stem}** — `{path.relative_to(ROOT)}`  \n{caption}"))
    display(Image(filename=str(path), width=width))
from morphonup import analysis, arms

ARM_LIST = ARMS or arms.GROUPS[GROUP]
TRAIN = [a for a in ARM_LIST if a in arms.ARMS]
K_DIR = run_dir(DATASET, K_FOLDS).parent
display(arms.table().set_index("arm").loc[ARM_LIST, ["label", "change", "needs_training"]])
full = jobs.progress(DATASET, K_FOLDS, SEEDS)
print("MorphoNup seeds finished:", int((full.status == "done").sum()), "of", len(SEEDS))
gpu, procs = jobs.gpu_status()
print("GPU:", gpu, "| processes:", procs or "none")
print("MorphoNup jobs running:", jobs.running_jobs() or "none")

## Train the arms

In [ ]:
if RUN_MODE == "background":
    LOGS = jobs.launch_arms(DATASET, K_FOLDS, SEEDS, TRAIN, allow_busy_gpu=ALLOW_BUSY_GPU, chains=CHAINS)
    print("launched", len(TRAIN), "arm(s) in", len(LOGS), "job(s); logs:")
    for log in LOGS:
        print(" ", log.relative_to(ROOT))
elif RUN_MODE == "inline":
    gpu, procs = jobs.gpu_status()
    if procs and not ALLOW_BUSY_GPU:
        raise RuntimeError("GPU in use: " + "; ".join(procs))
    import tensorflow as tf
    from morphonup.cv import run
    from morphonup.data import Dataset
    ds = Dataset(config.dataset_folder(DATASET))
    for device in tf.config.list_physical_devices("GPU"):
        tf.config.set_logical_device_configuration(
            device, [tf.config.LogicalDeviceConfiguration(memory_limit=config.gpu_memory_mb(len(ds.y)))])
    for arm in TRAIN:
        run(ds, DATASET, K_FOLDS, SEEDS, arm=arm)
else:
    print("results_only: no training started")

## Progress

In [ ]:
status = pd.concat([jobs.progress(DATASET, K_FOLDS, SEEDS, arm) for arm in TRAIN], ignore_index=True)
display(status.pivot(index="arm", columns="seed", values="status").loc[TRAIN])
for log in jobs.arm_logs(DATASET, K_FOLDS, SEEDS, TRAIN):
    print("==", log.name)
    print(jobs.tail(log, 6))

## Results

In [ ]:
analysis.refresh(K_DIR)
OUT = K_DIR / "analysis" / GROUP
table = OUT / "manuscript_table.md"
display(Markdown(table.read_text()) if table.exists() else Markdown("_No arm of this group has a finished seed yet._"))

In [ ]:
for extra in ("metrics_table.md", "comparison.md"):   # every metric; every statistic
    if (OUT / extra).exists():
        display(Markdown(f"**{extra}**\n\n" + (OUT / extra).read_text()))

In [ ]:
if (OUT / "figures").exists():
    for name in ('deltas', 'boxplot_accuracy', 'roc_pr_arms', 'delta_heatmap', 'metrics_panel', 'accuracy', 'discordance', 'fold_deltas', 'roc_pr_parts', 'training_curves', 'training_diagnostics', 'parameters'):
        if (OUT / "figures" / f"{name}.png").exists():
            show(OUT / "figures" / f"{name}.png")
else:
    print("No figures yet: train at least one arm (RUN_MODE = 'background').")

Per-seed and per-fold numbers: `seed_metrics.csv`, `fold_deltas.csv`, `comparison.csv`, `manuscript_table.csv` in the same folder. Each arm's own run figures are under `results/<dataset>/k<K>/<arm>/`.